# Cross-Scale Phase Coherence: K_nm Predictions vs PhysioNet Data

**Goal**: Test whether the SCPN K_nm matrix predicts the measured
cross-modality phase coherence between simultaneous physiological signals.

**Method**: Compute pairwise phase-locking values (PLV) between EEG bands,
ECG R-R intervals, and respiratory cycles from real multi-modal recordings.
Compare the measured PLV matrix to K_nm predictions.

**Why this matters**: Every existing SCPN notebook simulates Kuramoto with
K_nm parameters and shows sync emerges. That proves Kuramoto works, not
that K_nm is correct. This notebook asks: does K_nm predict real data?

In [ ]:
import numpy as np
from scipy import signal
from scipy.stats import pearsonr, spearmanr

from scpn_quantum_control.bridge import build_knm_paper27, OMEGA_N_16

## 1. Map Physiological Signals to SCPN Layers

| Signal | SCPN Layer | Frequency Band | Justification |
|--------|-----------|----------------|---------------|
| EEG delta (0.5-4 Hz) | L9 (memory) | Sleep slow oscillations | Hippocampal replay |
| EEG theta (4-8 Hz) | L5 (self/emotional) | Limbic rhythm | Emotional processing |
| EEG alpha (8-13 Hz) | L10 (boundary control) | Thalamic gating | Sensory gating |
| EEG beta (13-30 Hz) | L4 (cellular sync) | Cortical binding | Neural assembly |
| EEG gamma (30-80 Hz) | L7 (symbolic) | Conscious content | Feature binding |
| ECG R-R (0.8-1.2 Hz) | L4 (cellular sync) | Cardiac pacemaker | SA node |
| Respiration (0.15-0.4 Hz) | L6 (biosphere) | Autonomic rhythm | Brainstem |
| HRV-LF (0.04-0.15 Hz) | L6 (biosphere) | Baroreflex | Sympathovagal |
| HRV-HF (0.15-0.4 Hz) | L6 (biosphere) | RSA | Parasympathetic |

In [ ]:
# SCPN layer assignments for each signal
SIGNAL_LAYERS = {
    'eeg_delta': 9,
    'eeg_theta': 5,
    'eeg_alpha': 10,
    'eeg_beta': 4,
    'eeg_gamma': 7,
    'ecg_rr': 4,
    'resp': 6,
    'hrv_lf': 6,
    'hrv_hf': 6,
}

K16 = build_knm_paper27(L=16)
print(f'K_nm matrix: {K16.shape}')
print(f'K[4,5] = {K16[3,4]:.4f} (cellular-organismal)')
print(f'K[5,6] = {K16[4,5]:.4f} (organismal-biosphere)')
print(f'K[9,10] = {K16[8,9]:.4f} (memory-boundary)')

## 2. Generate Synthetic Multi-Modal Data

Real PhysioNet data requires `wfdb` (not installed). Instead, generate
synthetic signals with KNOWN cross-scale coupling strengths, then verify
the PLV extraction pipeline recovers them. Then show what K_nm predicts
and what would need to match in real data.

In [ ]:
def generate_coupled_oscillators(N, freqs, K, T=60.0, fs=256.0, noise=0.1, seed=42):
    """Kuramoto oscillators with given coupling, return timeseries."""
    rng = np.random.default_rng(seed)
    dt = 1.0 / fs
    n_steps = int(T * fs)
    phases = rng.uniform(0, 2*np.pi, N)
    timeseries = np.zeros((N, n_steps))
    
    for t in range(n_steps):
        for i in range(N):
            coupling = sum(K[i,j] * np.sin(phases[j] - phases[i]) for j in range(N))
            phases[i] += dt * (freqs[i] * 2 * np.pi + coupling + noise * rng.normal())
        timeseries[:, t] = np.sin(phases)
    return timeseries


def phase_locking_value(x, y, fs=256.0):
    """PLV between two signals via Hilbert transform."""
    phase_x = np.angle(signal.hilbert(x))
    phase_y = np.angle(signal.hilbert(y))
    return float(np.abs(np.mean(np.exp(1j * (phase_x - phase_y)))))


# 9 signals mapped to their SCPN layers
signal_names = list(SIGNAL_LAYERS.keys())
signal_layer_idx = [SIGNAL_LAYERS[s] - 1 for s in signal_names]  # 0-indexed
N = len(signal_names)

# Extract the sub-matrix of K_nm for these specific layers
K_predicted = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        K_predicted[i, j] = K16[signal_layer_idx[i], signal_layer_idx[j]]

# Frequencies for each signal (Hz)
freqs = np.array([2.0, 6.0, 10.0, 20.0, 40.0, 1.0, 0.25, 0.08, 0.25])

print('Generating synthetic coupled signals...')
# Scale K_predicted to reasonable coupling strength
K_scaled = K_predicted * 2.0  # scaling factor
ts = generate_coupled_oscillators(N, freqs, K_scaled, T=30.0, fs=256.0, noise=0.3)
print(f'Generated {N} signals, {ts.shape[1]} samples')

In [ ]:
# Compute PLV matrix
PLV = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i == j:
            PLV[i, j] = 1.0
        else:
            PLV[i, j] = phase_locking_value(ts[i], ts[j])

print('Phase-Locking Value matrix (from synthetic signals):')
print(f'{"":>12}', end='')
for s in signal_names:
    print(f'{s[:8]:>9}', end='')
print()
for i, s in enumerate(signal_names):
    print(f'{s[:12]:>12}', end='')
    for j in range(N):
        print(f'{PLV[i,j]:9.3f}', end='')
    print()

In [ ]:
# Compare: K_nm predicted vs measured PLV
# Extract upper triangle (excluding diagonal)
idx = np.triu_indices(N, k=1)
k_flat = K_predicted[idx]
plv_flat = PLV[idx]

r_pearson, p_pearson = pearsonr(k_flat, plv_flat)
r_spearman, p_spearman = spearmanr(k_flat, plv_flat)

print(f'\nK_nm vs PLV correlation (upper triangle, {len(k_flat)} pairs):')
print(f'  Pearson:  r={r_pearson:.4f}, p={p_pearson:.4e}')
print(f'  Spearman: r={r_spearman:.4f}, p={p_spearman:.4e}')
print()
print('INTERPRETATION:')
if p_pearson < 0.01:
    print(f'  K_nm predicts PLV structure (r={r_pearson:.3f}, p<0.01)')
    print('  BUT: synthetic data was generated WITH K_nm coupling.')
    print('  This only validates the pipeline, not the K_nm values.')
    print('  Real test: repeat with PhysioNet MIMIC-III or Sleep-EDF data.')
else:
    print(f'  No significant correlation (p={p_pearson:.3f})')
    print('  Pipeline may need frequency-matched band-pass filtering.')

In [ ]:
# What K_nm predicts for the TOP cross-scale couplings
print('K_nm PREDICTIONS for cross-modal phase coherence:')
print('(These are testable with real simultaneous recordings)')
print()
pairs = []
for i in range(N):
    for j in range(i+1, N):
        if signal_layer_idx[i] != signal_layer_idx[j]:  # cross-layer only
            pairs.append((signal_names[i], signal_names[j], K_predicted[i,j]))

pairs.sort(key=lambda x: -x[2])
print(f'{"Signal A":<15} {"Signal B":<15} {"K_predicted":>12} {"Expected PLV":>12}')
print('-' * 58)
for a, b, k in pairs[:15]:
    # PLV ~ tanh(K * scale) as rough mapping
    plv_est = np.tanh(k * 3)
    print(f'{a:<15} {b:<15} {k:12.4f} {plv_est:12.3f}')

In [ ]:
# Null model comparison: random K vs structured K_nm
print('\nNULL MODEL TEST: random coupling vs K_nm')
print('If K_nm has no predictive value, random K should fit equally well.\n')

rng = np.random.default_rng(42)
n_shuffles = 1000
r_null = []
for _ in range(n_shuffles):
    k_shuffled = k_flat.copy()
    rng.shuffle(k_shuffled)
    r_null.append(pearsonr(k_shuffled, plv_flat)[0])

r_null = np.array(r_null)
p_perm = np.mean(np.abs(r_null) >= np.abs(r_pearson))

print(f'  Observed r:    {r_pearson:.4f}')
print(f'  Null mean r:   {np.mean(r_null):.4f} +/- {np.std(r_null):.4f}')
print(f'  Permutation p: {p_perm:.4f}')
print(f'  K_nm is {"BETTER" if p_perm < 0.05 else "NOT BETTER"} than random coupling')
print()
print('TO MAKE THIS PUBLISHABLE:')
print('  1. pip install wfdb && download PhysioNet Sleep-EDF or MIMIC-III')
print('  2. Extract simultaneous EEG + ECG + respiration')
print('  3. Compute real PLV matrix')
print('  4. Compare real PLV to K_nm predictions')
print('  5. If r > 0.5 and p_perm < 0.01: K_nm predicts real physiology')

In [ ]:
import json
print('--- JSON RESULTS ---')
print(json.dumps({
    'n_signals': N,
    'n_pairs': len(k_flat),
    'r_pearson': round(r_pearson, 4),
    'p_pearson': float(f'{p_pearson:.4e}'),
    'r_spearman': round(r_spearman, 4),
    'p_perm': round(p_perm, 4),
    'null_mean_r': round(float(np.mean(r_null)), 4),
    'data_source': 'synthetic (pipeline validation only)',
    'next_step': 'repeat with PhysioNet real recordings',
}, indent=2))